In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from scipy.stats import mode
from sklearn.svm import SVC

In [2]:
import sys
sys.path.append("/Users/mariahloehr/IICD/IICD/feature_importance")

In [3]:
import locomp
from locomp import *
from locomp.MLmodels import *
from locomp.util_locomp import *
import itertools
import importlib
from sklearn.base import BaseEstimator, RegressorMixin, clone
import itertools
from functools import partial
import multiprocessing as mp
import re

import functions_case as il
import importlib

In [4]:
# Load data
df = pd.read_csv("/Users/mariahloehr/IICD/IICD/Cancer treatment/T47D.csv")

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Separate features and target
X = df.drop(columns=['phase'])
y = le.fit_transform(df['phase'])
perturbation_col = X.columns.get_loc("Metadata_well")

feature_names = X.columns.tolist()
feature_pairs = list(itertools.combinations(range(X.shape[1]), 2))
X = X.to_numpy()
#y = y.to_numpy()

In [5]:
DecisionTreeClass = DecisionTreeClassifier(max_depth = 50, 
                                 max_features=10,
                                 random_state=949
                                 )

In [6]:
xgbclass = GradientBoostingClassifier(
        n_estimators=10,       # fixed boosting rounds
        learning_rate=0.1, # hyperparameters from XGB model
        max_depth=7,
        random_state=949
    )

In [7]:
# Define RBF-kernel SVM
svmclass = SVC(kernel='rbf', C = 400, gamma = 0.01, probability=True, random_state=949
              )

In [8]:
import time
import tracemalloc

def benchmark_model(model, X, y, n_runs=10, n_samples=500, n_features=10):
    times, mems = [], []

    for _ in range(n_runs):
        # Subsample rows and features
        idx_rows = np.random.choice(X.shape[0], n_samples, replace=False)
        idx_cols = np.random.choice(X.shape[1], n_features, replace=False)
        X_sub, y_sub = X[idx_rows][:, idx_cols], y[idx_rows]

        # Measure memory + time
        tracemalloc.start()
        t0 = time.perf_counter()
        model.fit(X_sub, y_sub)
        preds = model.predict(X_sub)
        t1 = time.perf_counter()
        current, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()

        times.append(t1 - t0)          # seconds
        mems.append(peak / 1e6)        # MB

    return {
        "mean_time": np.mean(times),
        "std_time": np.std(times),
        "mean_mem": np.mean(mems),
        "std_mem": np.std(mems),
    }


In [12]:
# Models to test
models = {DecisionTreeClass, xgbclass, svmclass
}

# Run benchmarks
n_runs = 10   # change to 100 for longer benchmark
results = []

for model in models:
    print(f"Benchmarking {model} ...")
    stats = benchmark_model(model, X, y, n_runs=n_runs)
    results.append([model, stats["mean_time"], stats["std_time"],
                    stats["mean_mem"], stats["std_mem"]])

# Collect results in DataFrame
df = pd.DataFrame(results, columns=["Model", "Mean Time (s)", "Std Time (s)",
                                    "Mean Memory (MB)", "Std Memory (MB)"])
print("\nBenchmark results:")
df.to_string(index=False)

df

Benchmarking DecisionTreeClassifier(max_depth=50, max_features=10, random_state=949) ...
Benchmarking GradientBoostingClassifier(max_depth=7, n_estimators=10, random_state=949) ...
Benchmarking SVC(C=400, gamma=0.01, probability=True, random_state=949) ...

Benchmark results:


,Model,Mean Time (s),Std Time (s),Mean Memory (MB),Std Memory (MB)
0,"DecisionTreeClassifier(max_depth=50, max_featu...",0.005792,0.001920,0.055359,0.000139
1,([DecisionTreeRegressor(criterion='friedman_ms...,0.165864,0.043900,0.120585,0.003246
2,"SVC(C=400, gamma=0.01, probability=True, rando...",0.017375,0.004893,0.097535,0.003114
